# NeuralProphet - Modelo Aplicado a demanda energética Alemania
hacer modelo SARIMA, en dicho apartado hacer la validación con la ACF y la PACF, periodograma y demas


In [ ]:
# Modelado
#import tensorflow as tf # no se requiere
#from tensorflow import keras # no se requiere
#from keras.models import Sequential # no se requiere
#from keras import Input # no se requiere
#from keras.layers import Dense, SimpleRNN # no se requiere

import os
os.environ["PYTHONHASHSEED"] = "42"

import random
random.seed(42)

import numpy as np
np.random.seed(42)

import torch
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# procesamiento
import pandas as pd
import json
# ml, procesamiento
#import sklearn
from sklearn.model_selection import TimeSeriesSplit
#from sklearn.model_selection import train_test_split # no se requiere
from sklearn.preprocessing import MinMaxScaler 
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sktime.performance_metrics.forecasting import MeanAbsolutePercentageError
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.model_selection import GridSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from neuralprophet import NeuralProphet, set_log_level
# analisis series de tiempo
#from statsmodels.graphics.tsaplots import plot_acf, plot_pacf # no se requiere
#from statsmodels.tsa.stattools import adfuller # no se requiere
from statsmodels.graphics.tsaplots import acf, pacf
# optimización, warnings
import optuna
import warnings
import optuna.visualization as vis
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split

# visual
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
#plt.style.use("seaborn-v0_8-darkgrid")
#sns.set_palette("viridis")

# semilla para reproductibilidad
# configuración de reproductbilidad
np.random.seed(42)
from optuna.samplers import TPESampler # reproductibilidad con optuna
from scipy.signal import periodogram



Se cargan los valores de consumo a nivel semanal. La semana en este caso se toma el primer día como lunes, es fecha muestra el promedio de la semana y se formatean las fechas en formato date. El orden de las observaciones es de $10^6$

In [ ]:
date_close =  pd.read_csv("../../../data/csv/EC_GER_Weekly.csv")
df_data_close = pd.DataFrame(date_close.copy()) # copia del df de precios de cierre como df
df_data_close["Start date"] = pd.to_datetime(df_data_close["Start date"])
df_data_close = df_data_close.sort_values(by="Start date", ascending=True) # se ordena por fecha ascendente
df_data_close.head()


Se renombran las columnas de la siguiente forma:
- Start date: ds
- value: y

Según documentación oficial

In [ ]:
df_data_close = pd.DataFrame(date_close.copy()) # copia del df de precios de cierre como df
#df_data_close["Date"] = pd.to_datetime(df_data_close["Date"])
#df_data_close["Date"] = pd.to_datetime(df_data_close["Start date"])
df_data_close["Start date"] = pd.to_datetime(df_data_close["Start date"])
df_data_close = df_data_close.sort_values(by="Start date", ascending=True) # se ordena por fecha ascendente
#df_data_close = df_data_close.set_index("Date")
df_data_close = df_data_close.rename(columns={
    "Start date": "ds",
    "value": "y"
})
df_data_close.head()


In [ ]:
# # se grafican los datos de consumo de forma semanal
plt.figure(figsize=(12,6))
plt.plot(df_data_close["ds"], df_data_close["y"], label="demanda", color="black")
plt.xlabel("Fecha")
plt.ylabel("Consumo")
plt.legend()

# esta parte es para que las fechas en el eje x se vean mejor
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.AutoDateLocator()) # se ajusta el espaciado de las fechas
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d')) # se formatea la fechas
#plt.grid(True)
plt.tight_layout()# se ajusta el layout para que no se corten los elementos
plt.show()

In [ ]:
# partición temporal
percentage = 0.9
folds_data_size = int(len(df_data_close) * percentage)
folds_data = df_data_close.iloc[:folds_data_size]
val_data = df_data_close.iloc[folds_data_size:]
start_val_data = val_data["ds"].values[0]

print(start_val_data)

gráfica folds/val

In [ ]:
# se grafican los datos folds/val
plt.figure(figsize=(12,6))
plt.plot(folds_data["ds"], folds_data["y"], label="Folds", color="black")
plt.plot(val_data["ds"], val_data["y"], label="Val", color="orange")
plt.axvline(start_val_data, label="fecha folds/val", color="black")
#plt.plot(val_data.index, val_data["NVDA_log_diff"], label="Serie Transformada", color="orange")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.legend()

# esta parte es para que las fechas en el eje x se vean mejor
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.AutoDateLocator()) # se ajusta el espaciado de las fechas
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d')) # se formatea la fechas
#plt.grid(True)
plt.tight_layout()# se ajusta el layout para que no se corten los elementos
plt.show()

In [ ]:
print(len(folds_data))
print(len(val_data))

In [ ]:
def check_stationarity(ts):
    result = adfuller(ts, autolag='AIC')
    p_value = result[1]
    print(f"ADF Statistic: {result[0]:.16f}")
    print(f"p-value: {p_value:.16f}")
    print('Estacionaria' if p_value < 0.05 else 'No estacionaria')

def plot_acf_pacf(ts, n_lags=200, title_graph=["ACF", "PACF"]):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17,6))

    plot_acf(
        ts,
        ax=ax1,
        lags=n_lags,
        alpha=0.05,
        vlines_kwargs={"colors": "black"},
        marker="o",
        color="black",
        markersize=4,
        title=title_graph[0]
    )

    plot_pacf(
        ts,
        ax=ax2,
        lags=n_lags,
        method="ywm",
        alpha=0.05,
        vlines_kwargs={"colors": "black"},
        marker="o",
        color="black",
        markersize=4,
        title=title_graph[1]
    )

    # cambiar color de intervalos de confianza
    for ax in [ax1, ax2]:
        for collection in ax.collections:
            collection.set_facecolor("lightgray")
            collection.set_alpha(0.6)

    return plt.show()

check_stationarity(folds_data["y"])
plot_acf_pacf(folds_data["y"])

### Creación de Folds
Para el conjunto de datos X_Folds se definen una serie de "particiones" que ayudarán a determinar si el modelo encontrado se ajusta de forma consistente en diversas franjas de tiempo, ayudando a evitar inferir correlaciones espurias 

In [ ]:
# visualizacion de los folds 
# se muestra de forma visual como funcionan los folds
# si no se define el tamaño de test, en la primera iteracion se toma igual cantidad de datos para train que para test
# por eso se toman los folds de prueba como de la longitud de datos de validación
size_test = int(len(val_data))
tscv = TimeSeriesSplit(
    n_splits=6
    #test_size=size_test, 
    #gap=gap
)

folds, indexes, types = [], [], []

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(folds_data)):
    folds.extend([fold_idx] * len(train_idx)) 
    indexes.extend(train_idx)
    types.extend(['Train'] * len(train_idx))

    folds.extend([fold_idx] * len(test_idx))
    indexes.extend(test_idx)
    types.extend(['Test'] * len(test_idx))

# figura con plotly
fig = go.Figure()

# data train
fig.add_trace(go.Scatter(
    x=indexes, y=folds,
    mode='markers',
    marker=dict(color=['blue' if t == 'Train' else 'orange' for t in types], size=6),
    text=types,
    hovertemplate='Index: %{x}<br> Fold: %{y}<br> Tipo: %{text}<extra></extra>'
))

fig.update_layout(
    title='Folds',
    xaxis_title='Indices serie',
    yaxis_title='Fold',
    yaxis=dict(autorange='reversed'),
    legend_title='Tipo de conjunto',
    template='plotly_white',
    height=400
)
fig.show()

# Función de optimización

Mediante el uso de Optuna (una librería basada en técnicas de optimización bayesiana secuencial) se define una función de optimización que utiliza validación cruzada temporal para minimizar una métrica de error (en este caso, el MAPE*).

La función está dada de la siguiente forma: primero, se definen los hiperparámetros de la rnn: número de rezagos, unidades recurrentes, tasa de aprendizaje, tamaño de lote y número de épocas.   Para cada uno de estos parámetros se especifican rangos o conjuntos discretos de valores que Optuna explorará con el fin de encontrar la combinación que minimice el error promedio.

Posteriormente, se inicializan listas que almacenarán las métricas de evaluación obtenidas en cada fold (MAPE, MAE y RMSE).  
Durante cada iteración del proceso de validación cruzada, se realiza lo siguiente:
0. se definen los conjuntos train/test de acuerdo con ese fold
1. Se instancia un nuevo modelo RNN con el conjunto de hiperparámetros seleccionados para el trial actual.  
2. Se entrena el modelo sobre el conjunto de entrenamiento definido para ese fold.  
3. Se evalúa el modelo sobre el conjunto de prueba correspondiente, calculando las métricas de error.  

Este proceso se repite para los *k* folds (en este caso, 5), produciendo un conjunto de errores por fold:

- Fold₁ → MAPE₁  
- Fold₂ → MAPE₂  
- Fold₃ → MAPE₃  
- Fold₄ → MAPE₄  
- Fold₅ → MAPE₅  

Así se obtiene el MAPE promedio de los cinco folds:$$ \text{MAPE}_{prom} = \frac{1}{5} \sum_{i=1}^{5} \text{MAPE}_i $$

Este valor se devuelve como resultado de la función objetivo, y representa el desempeño promedio del modelo bajo ese conjunto de hiperparámetros. Optuna repite este proceso en múltiples trials, cada uno con una combinación diferente de hiperparámetros, y selecciona el conjunto que minimiza el MAPE promedio obtenido.


In [ ]:
# como no hay regresores (variables exogenas)
def objective_neuralprophet(trial):

    params = {
        "n_changepoints": trial.suggest_int("n_changepoints", 1, int(len(folds_data) * 0.5) ), # cantidad de puntos de cambio de tendencia
        "n_lags": trial.suggest_categorical("n_lags", [1, 2, 3, 4, 8, 12, 52]), # lags a tener en cuenta para la AR-net - revisar esto!
        "n_forecasts": 1,
        "yearly_seasonality": True, #estacionalidad anual (52 semanas)
        "weekly_seasonality": False,
        "daily_seasonality": False,
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.05, log=True),
        "epochs": trial.suggest_int("epochs", 50, 200),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128])
    }

    mapes, maes, rmses, smapes = [], [], [], []
    smape_fn = MeanAbsolutePercentageError(symmetric=True)

    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(folds_data)):
        torch.manual_seed(42)
        np.random.seed(42)
        random.seed(42)

        df_train = folds_data.iloc[train_idx]
        df_test  = folds_data.iloc[test_idx]

        model = NeuralProphet(**params)

        # se añaden los días festivos de Alemania

        model = model.add_country_holidays(country_name="DE")
        model.fit(df_train, freq="W-MON", progress="off")

        # combinar train + test para mantener continuidad temporal
        df_combined = pd.concat([df_train, df_test])

        forecast = model.predict(df_combined)

        # quedarnos solo con el rango del test
        forecast_test = forecast.iloc[-len(df_test):]

        y_pred = forecast_test["yhat1"].values
        y_true = df_test["y"].values


        mae  = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mape = mean_absolute_percentage_error(y_true, y_pred)
        smape = smape_fn(y_true, y_pred)

        mapes.append(mape)
        maes.append(mae)
        rmses.append(rmse)
        smapes.append(smape)

        trial.set_user_attr(f"fold_{fold_idx+1}", {
            "mae": float(mae),
            "mape": float(mape),
            "rmse": float(rmse),
            "smape": float(smape)
        })

        trial.report(np.mean(mapes), step=fold_idx)

        #if trial.should_prune():
        #    raise optuna.TrialPruned()

    mean_mape = float(np.mean(mapes))
    mean_mae = float(np.mean(maes))
    mean_rmse = float(np.mean(rmses))
    mean_smape = float(np.mean(smapes))

    # guardar como user attrs
    trial.set_user_attr("mape_mean", mean_mape)
    trial.set_user_attr("mae_mean", mean_mae)
    trial.set_user_attr("rmse_mean", mean_rmse)
    trial.set_user_attr("smape_mean", mean_smape)
    

    return mean_mape

study_np = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=42)
)

study_np.optimize(objective_neuralprophet, n_trials=50)

print("Mejores parámetros:")
print(study_np.best_params)

Se extraen los resultados del mejor trial

In [ ]:
# Extraer resultados del mejor trial
best_trial = study_np.best_trial
print("Mejor trial:", best_trial.number)
print("MAPE promedio:", best_trial.value)
print("Atributos guardados:")
print(best_trial.user_attrs)

se obtiene todo el listado de los trials ejecutados y se crea un df que muestra los errores por fold y por trial

In [ ]:

# info de los trials 
# se extraen los trials
trials = study_np.get_trials()
# creación del df a nivel de registro
# el esquema del df se infiere a partir de las claves almacenadas en cada registro (las claves no varían, dado como se guardaron)
rows = [] # se usará para guardar la info a nivel de registro
for trial in trials:
    trial_number = trial.number # número de trial: trial 1, 2,...
    attrs = trial.user_attrs # los valores de las métricas se definen en la función como 
    # un atributo de usuario, definido como un diccionario
    # se recorren ahora las claves y los valores de cada diccionario
    for key, value in attrs.items():
        if key.startswith("fold_") and isinstance(value, dict): 
            # se examina si cada clave del diccionario empieza con fold_. Recordar que en la función
            # se guardó un diccionario con el nombre fold_i_metrics, que guarda la info de cada fold y cada trial.
            # Se mira entonces si la clave es de esa forma y si el valor de esa clave es un diccionario
            rows.append({
                "trial": trial_number,
                "fold": int(key.replace("fold_", "")),   # extraer número
                "mape": value["mape"],
                "mae": value["mae"],
                "rmse": value["rmse"],
                "smape": value["smape"]
            })

df_metrics = pd.DataFrame(rows)
df_metrics.head()


In [ ]:
# descripción general del df de métricas
df_stats = df_metrics.describe()
df_stats[["mape", "mae", "rmse", "smape"]]

### Distribución por errores promedio
se muestran las distribuciones de los errores promedio por trial

In [ ]:
df_metrics_avg = df_metrics.copy()
df_metrics_avg = df_metrics_avg.groupby(
    "trial", as_index=False 
)[["mape", "mae", "rmse", "smape"]].mean()
df_metrics_avg.head()

In [ ]:
# histograma erorres promedio
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, metric in zip(axes, ["mape", "mae", "rmse", "smape"]):
    sns.histplot(df_metrics_avg[metric], kde=True, ax=ax, color="gray")
    ax.set_title(f"Distribución de {metric.upper()} promedio por trial")
    ax.axvline(df_metrics_avg[metric].mean(), color='red', linestyle='--', label='Media')
    ax.axvline(df_metrics_avg[metric].median(), color='green', linestyle='--', label='Mediana')
    ax.legend()

plt.tight_layout()
plt.show()

### Distribución de errores por fold y por trial
se muestran los valores de las métricas de error por cada fold y por cada trial

In [ ]:
# histogramas de los errores por trial y por fold
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, metric in zip(axes, ["mape", "mae", "rmse", "smape"]):
    sns.histplot(df_metrics[metric], kde=True, ax=ax, color="gray")
    ax.set_title(f"Distribución de {metric.upper()}")
    ax.axvline(df_metrics[metric].mean(), color='red', linestyle='--', label='Media')
    ax.axvline(df_metrics[metric].median(), color='green', linestyle='--', label='Mediana')
    """ if metric == 'mape':
        medians_df['mape'] = df_metrics[metric].median() """
    ax.legend()

plt.tight_layout()
plt.show()


Se ordena por métrica 

In [ ]:
# Ordenar por cada métrica para tomar los mejores conjuntos de parametros
df_mape_sorted = df_metrics.sort_values("mape", ascending=True).reset_index(drop=True)
df_mae_sorted = df_metrics.sort_values("mae", ascending=True).reset_index(drop=True)
df_rmse_sorted = df_metrics.sort_values("rmse", ascending=True).reset_index(drop=True)
df_smape_sorted = df_metrics.sort_values("smape", ascending=True).reset_index(drop=True)

print("Mejores Folds por MAPE:")
print(df_mape_sorted.tail())

print("\Mejores Folds por MAE:")
print(df_mae_sorted.tail())

print("\Mejores Folds por RMSE:")
print(df_rmse_sorted.tail())

print("\Mejores Folds por sMAPE:")
print(df_smape_sorted.tail())

Se crea df que obtiene los hiperparametros por fold junto con las metricas asociadas, se toman los 3 que dejan menor mape promedio para comparar y luego se toma uno solo

In [ ]:
for trial in trials:
    print(trial.user_attrs)

In [ ]:
df_trials = {   
    "trial" :[],
    "n_lags": [],
    "n_changepoints": [],
    "learning_rate": [],
    "epochs": [],
    "batch_size": [],
    "mape_mean": [],
    "mae_mean": [],
    "rmse_mean": [],
    "smape_mean": []
}


for trial in trials:
    for key in df_trials.keys():

        if key == "trial":
            df_trials[key].append(trial.number)

        elif key in ["mape_mean", "mae_mean", "rmse_mean", "smape_mean"]:
            df_trials[key].append(trial.user_attrs[key])

        else: 
            df_trials[key].append(trial.params[key])
   
df_params_models = pd.DataFrame(df_trials)
df_params_models = df_params_models.sort_values("mape_mean") # se ordenan conjuntos de hiperparametros que dejan menor mape promedio y se toma
# el menor
df_params_models.head(5)

In [ ]:
# se exporta el df con los mejores modelos 
df_params_models.head(3).to_csv("../../../data/best_train_models/best_models_NeuralProphet.csv")

In [ ]:
# se fija el modelo 
#metrics = model.fit(folds_data, freq="D")
model = NeuralProphet(**study_np.best_params)
model = model.add_country_holidays(country_name="DE")
model.fit(folds_data, freq="W-MON", progress="off")


In [ ]:
# se hacen predicciones 
#future = model.make_future_dataframe(
#    folds_data,
#    periods=len(val_data),
#    n_historic_predictions=True
#)
#forecast = model.predict(future)

# se extraen predicciones del test
#forecast_test = forecast.iloc[-len(val_data):]

#y_pred = forecast_test["yhat1"].values
y_true = val_data["y"].values

df_combined = pd.concat([folds_data, val_data])
forecast = model.predict(df_combined)
forecast_test = forecast.iloc[-len(val_data):]
y_pred = forecast_test["yhat1"].values

In [ ]:
smape_fn = MeanAbsolutePercentageError(symmetric=True)
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true))
smape = smape_fn(y_true, y_pred)

print(f"MAE: {mae}")
print(f"RMSE: {rmse}")
print(f"MAPE: {mape*100:.4f}%")
print(f"sMAPE: {smape*100:.4f}%")


In [ ]:
plt.figure(figsize=(10,6))
plt.plot(folds_data["ds"], folds_data["y"], label="Folds", color="gray")
plt.plot(val_data["ds"], y_true, label="Real", color="orange")
plt.plot(val_data["ds"], y_pred, label="Predicción", color="red")
plt.legend()
plt.show()


Se exportan las predicciones en json para poder calcular luego la predicción conjunta

In [ ]:
arr_preds = []
for index, value in enumerate(df_data_close[folds_data_size:]["ds"]):
    dict_pred = {
        "Date": value.strftime("%Y-%m-%d"), # convierte a la fecha a string para poder exportar a json
        "pred" : float(y_pred[index])
    }
    arr_preds.append(dict_pred)
arr_preds

In [ ]:
# Cálculo de métricas agregadas
dict_to_export = {
    "mape_val": float(mape),
    "mae_val": float(mae),
    "smape_val": float(smape),
    "rmse_val": float(rmse),
    "predictions": arr_preds
}
with open ("../../../data/preds_energy_models/neuralprophet_preds.json", "w") as json_file:
    json.dump(dict_to_export, json_file, indent=4)

validacion adicional: se ejecuta varias veces el modelo final y se generan las predicciones con el fin de validar que los rangos de variación de las métricas de error sean consistentes

In [ ]:
# se fija el modelo a los datos de folds
n_iters = 10 # cantidad de veces en las que se instancia el modelo 
preds_dict = []

for i in range(n_iters):
    model = NeuralProphet(**study_np.best_params)
    model = model.add_country_holidays(country_name="DE")
    model.fit(folds_data, freq="W-MON", progress="off")
    
    y_true = val_data["y"].values

    df_combined = pd.concat([folds_data, val_data])
    forecast = model.predict(df_combined)
    forecast_test = forecast.iloc[-len(val_data):]
    y_pred = forecast_test["yhat1"].values

    # métricas de error
    smape_fn = MeanAbsolutePercentageError(symmetric=True)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    smape = smape_fn(y_true, y_pred)

    preds_dict.append({
        "iteracion": i,
        "mape": mape * 100,
        "rmse": rmse,
        "mae": mae,
        "smape": smape * 100,
        "preds": y_pred
    })